# Building models with `nn.Module`

Subclass `nn.Module`, register layers in `__init__`, and implement `forward`. Parameters registered as modules or `nn.Parameter` are picked up by `model.parameters()` and sent to the GPU by `model.to(device)`.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from collections import OrderedDict
import matplotlib.pyplot as plt

torch.manual_seed(0)


## 1. A first module


In [ ]:
class MySimpleNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(in_features=10, out_features=5)
        self.activation = nn.ReLU()

    def forward(self, x):
        x = self.layer1(x)
        x = self.activation(x)
        return x


model = MySimpleNetwork()
print(model)
print("output shape:", model(torch.randn(4, 10)).shape)


## 2. `nn.Parameter` vs a plain tensor

Only `nn.Parameter` (and submodules) appear in `named_parameters()` and get gradients by default.


In [ ]:
class MyCustomModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.my_weight = nn.Parameter(torch.randn(5, 2))
        self.my_info = torch.tensor([1.0, 2.0])  # buffer-like, not trained

    def forward(self, x):
        return x @ self.my_weight


module = MyCustomModel()
for name, param in module.named_parameters():
    print(name, param.shape, "requires_grad =", param.requires_grad)


## 3. Linear regression as a module


In [ ]:
class SimpleModel(nn.Module):
    def __init__(self, input_features, output_features):
        super().__init__()
        self.linear_layer = nn.Linear(input_features, output_features)

    def forward(self, x):
        return self.linear_layer(x)


model = SimpleModel(10, 1)
dummy_input = torch.randn(5, 10)
output = model(dummy_input)
print("output:\n", output)
for name, param in model.named_parameters():
    print(f"{name}: {tuple(param.shape)}")


## 4. MLP: write `forward` yourself


In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.layer1(x)
        x = self.activation(x)  # or F.relu(x)
        x = self.layer2(x)
        return x


mlp = SimpleMLP(20, 32, 3)
print(mlp(torch.randn(8, 20)).shape)


## 5. The same MLP with `nn.Sequential`


In [ ]:
in_size, hidden_size, out_size = 784, 128, 10

model_v1 = nn.Sequential(
    nn.Linear(in_size, hidden_size),
    nn.ReLU(),
    nn.Linear(hidden_size, out_size),
)

model_v2 = nn.Sequential(
    OrderedDict(
        [
            ("fc1", nn.Linear(in_size, hidden_size)),
            ("relu1", nn.ReLU()),
            ("fc2", nn.Linear(hidden_size, out_size)),
        ]
    )
)

print(model_v2.relu1)
print("batch output:", model_v1(torch.randn(64, in_size)).shape)


## 6. Losses

- **CrossEntropyLoss**: classification. Inputs are **logits** (raw scores), targets are class **indices**. Softmax is included.
- **MSELoss**: regression.
- **BCEWithLogitsLoss**: binary classification with a single logit per sample.


In [ ]:
loss_fn_ce = nn.CrossEntropyLoss()
predictions_logits = torch.randn(3, 5, requires_grad=True)
targets_classes = torch.tensor([1, 0, 4])
loss_ce = loss_fn_ce(predictions_logits, targets_classes)
print("cross-entropy:", loss_ce.item())


## 7. One training step on dummy batches


In [ ]:
num_classes = 10
model = nn.Linear(784, num_classes)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

dummy_dataset = [
    (torch.randn(64, 784), torch.randint(0, num_classes, (64,))) for _ in range(5)
]

model.train()
for batch_idx, (data, target) in enumerate(dummy_dataset):
    optimizer.zero_grad()
    logits = model(data)
    loss = loss_fn(logits, target)
    loss.backward()
    optimizer.step()
    if batch_idx % 2 == 0:
        print(f"batch {batch_idx}, loss: {loss.item():.4f}")


## 8. Tiny binary classifier and a loss curve


In [ ]:
class SimpleNet(nn.Module):
    def __init__(self, input_size, hidden_size, out_size):
        super().__init__()
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(hidden_size, out_size)

    def forward(self, x):
        return self.layer2(self.activation(self.layer1(x)))


model = SimpleNet(2, 10, 1)
dummy_input = torch.randn(32, 2)
dummy_labels = torch.randint(0, 2, (32, 1)).float()

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.05)

epochs = 40
history = []
for _ in range(epochs):
    optimizer.zero_grad()
    outputs = model(dummy_input)
    loss = criterion(outputs, dummy_labels)
    history.append(loss.item())
    loss.backward()
    optimizer.step()

plt.plot(history)
plt.xlabel("epoch")
plt.ylabel("BCE with logits")
plt.title("Training loss")
plt.show()
